COBA 1

In [ ]:
# IMAGE CLASSIFICATION PIPELINE (AUTO LABELING)
# Menggunakan Feature Extraction + Clustering

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import os
from tqdm import tqdm


# 1. Directory Configuration
base_dir = os.path.join(os.getcwd(), "dataset")  # otomatis menunjuk ke folder 'dataset'
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

# 2. Load Data dan Ekstrak Fitur
feature_extractor = EfficientNetB0(weights="imagenet", include_top=False, pooling="avg", input_shape=(224, 224, 3))

def extract_features(directory):
    features = []
    filenames = []
    for file in tqdm(os.listdir(directory)):
        file_path = os.path.join(directory, file)
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        try:
            img = load_img(file_path, target_size=(224, 224))
            img_array = img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)
            feature = feature_extractor.predict(img_array, verbose=0)
            features.append(feature.flatten())
            filenames.append(file)
        except Exception as e:
            print(f" Error processing {file}: {e}")
            continue
    return np.array(features), filenames

print(" Ekstraksi fitur train images...")
train_features, train_filenames = extract_features(train_dir)
print(" Fitur train:", train_features.shape)


# 3. Clustering untuk Auto Labeling
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)

n_clusters = 15  # jumlah kelas
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(train_features_scaled)

unique, counts = np.unique(cluster_labels, return_counts=True)
print(" Distribusi label otomatis:", dict(zip(unique, counts)))

auto_labels = pd.DataFrame({
    'filename': train_filenames,
    'label': cluster_labels.astype(str)
})
auto_labels.to_csv(os.path.join(base_dir, "auto_train_labels.csv"), index=False)
print(" File auto_train_labels.csv berhasil dibuat di folder 'dataset'.")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
🔍 Ekstraksi fitur train images...


100%|██████████| 4257/4257 [23:34<00:00,  3.01it/s]


✅ Fitur train: (4257, 1280)
📊 Distribusi label otomatis: {0: 191, 1: 354, 2: 295, 3: 378, 4: 283, 5: 200, 6: 447, 7: 386, 8: 433, 9: 155, 10: 248, 11: 178, 12: 68, 13: 413, 14: 228}
✅ File auto_train_labels.csv berhasil dibuat di folder 'dataset'.


In [ ]:

# 4. Train-Validation Split
train_df, val_df = train_test_split(auto_labels, test_size=0.2, stratify=auto_labels['label'], random_state=42)

# 5. Data Generator
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

# 6. Model (Transfer Learning)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# freeze bbrp layer awal
for layer in base_model.layers[:200]:
    layer.trainable = False
for layer in base_model.layers[200:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_clusters, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True)
]

# 7. Training
print(" Mulai Training Model...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=callbacks
)

# 8. Fine-Tuning
base_model.trainable = True
for layer in base_model.layers[:150]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print(" Fine-tuning model...")
fine_tune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

# ===============================================
# 9. Evaluasi
# ===============================================
val_loss, val_acc = model.evaluate(val_generator)
print(f" Validation Accuracy: {val_acc:.4f}")

# ===============================================
# 10. Prediksi Test Set
# ===============================================
test_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_df = pd.DataFrame({'filename': test_files})
test_df['filepath'] = test_df['filename'].apply(lambda x: os.path.join(test_dir, x))

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col=None,
    target_size=(224, 224),
    class_mode=None,
    batch_size=32,
    shuffle=False
)

predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)

# 11. Submission
submission = pd.DataFrame({
    'filename': test_df['filename'],
    'label': predicted_classes
})
submission.to_csv(os.path.join(base_dir, 'submission.csv'), index=False)
print(" submission.csv berhasil dibuat di folder 'dataset'!")


Found 3405 validated image filenames belonging to 15 classes.
Found 852 validated image filenames belonging to 15 classes.
🚀 Mulai Training Model...
Epoch 1/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 261s 2s/step - accuracy: 0.0846 - loss: 2.6887 - val_accuracy: 0.1045 - val_loss: 2.6387
Epoch 2/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 222s 2s/step - accuracy: 0.0990 - loss: 2.6609 - val_accuracy: 0.0833 - val_loss: 2.6391
Epoch 3/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 220s 2s/step - accuracy: 0.1001 - loss: 2.6595 - val_accuracy: 0.1045 - val_loss: 2.6282
Epoch 4/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 227s 2s/step - accuracy: 0.1063 - loss: 2.6304 - val_accuracy: 0.0469 - val_loss: 3.5726
Epoch 5/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 225s 2s/step - accuracy: 0.1181 - loss: 2.5867 - val_accuracy: 0.0481 - val_loss: 6.3820
Epoch 6/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 166s 2s/step - accuracy: 0.1304 - loss: 2.5583 - val_accuracy: 0.1620 - val_loss: 2.4479
Epoch 7/25
107/107 ━━━━━━━━━━━━━━━━━━━━ 155s 1s/step - accuracy: 0.1310 - los

COBA 2

In [ ]:
# IMAGE CLASSIFICATION PIPELINE (AUTO LABELING + TRANSFER LEARNING)

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageStat

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# 1. Setup Paths
base_dir = os.path.join(os.getcwd(), "dataset")
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

print("Base dir:", base_dir)

# 2. Data Cleaning – hapus gambar kosong atau putih polos
def is_blank(image_path, threshold=250):
    """Deteksi gambar kosong/putih polos."""
    try:
        img = Image.open(image_path).convert("L")  # grayscale
        stat = ImageStat.Stat(img)
        mean = stat.mean[0]
        return mean > threshold  # terlalu putih = kosong
    except Exception:
        return True  # kalau error, anggap invalid

removed = 0
for root, _, files in os.walk(train_dir):
    for file in files:
        path = os.path.join(root, file)
        if is_blank(path):
            os.remove(path)
            removed += 1

print(f" Removed {removed} blank/white images.")

# 3. Pretrained Model (Feature Extractor dari ImageNet)
feature_extractor = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
print(" Loaded ResNet50 as feature extractor")

# 4. Ekstraksi Fitur dari Dataset
def extract_features(directory):
    features = []
    filenames = []
    for file in tqdm(os.listdir(directory)):
        file_path = os.path.join(directory, file)
        if not file.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        try:
            img = load_img(file_path, target_size=(224, 224))
            img_array = img_to_array(img)
            img_array = np.expand_dims(img_array, axis=0)
            img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
            feature = feature_extractor.predict(img_array, verbose=0)
            features.append(feature.flatten())
            filenames.append(file)
        except Exception as e:
            print(f"⚠️ Error processing {file}: {e}")
            continue
    return np.array(features), filenames

print("🔹 Extracting train features...")
train_features, train_filenames = extract_features(train_dir)
print("Train features shape:", train_features.shape)

# 5. Clustering untuk Auto Labeling
scaler = StandardScaler()
train_features_scaled = scaler.fit_transform(train_features)

n_clusters = 15  # ubah sesuai perkiraan jumlah kelas menu
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(train_features_scaled)

# Simpan hasil auto-label
auto_labels = pd.DataFrame({
    'filename': train_filenames,
    'label': cluster_labels.astype(str)
})
auto_labels.to_csv(os.path.join(base_dir, "auto_train_labels.csv"), index=False)
print(" Auto labeling done! Saved as auto_train_labels.csv")


Base dir: e:\ACTION DATA MINING\action_datamining\dataset
✅ Removed 160 blank/white images.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 24s 0us/step
✅ Loaded ResNet50 as feature extractor
🔹 Extracting train features...


100%|██████████| 4097/4097 [19:31<00:00,  3.50it/s]


Train features shape: (4097, 2048)
✅ Auto labeling done! Saved as auto_train_labels.csv


In [ ]:
# 6. Split Data Train-Validation
train_df, val_df = train_test_split(auto_labels, test_size=0.2, stratify=auto_labels['label'], random_state=42)

# 7. Data Generator
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

val_generator = datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=train_dir,
    x_col='filename',
    y_col='label',
    target_size=(224, 224),
    class_mode='categorical',
    batch_size=32
)

# 8. Model Transfer Learning
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:100]:
    layer.trainable = False
for layer in base_model.layers[100:]:
    layer.trainable = True

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(n_clusters, activation='softmax')
])

model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True)
]

print(" Training model...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    callbacks=callbacks
)

# 9. Fine-tuning Model
base_model.trainable = True
for layer in base_model.layers[:80]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    ModelCheckpoint(os.path.join(base_dir, 'best_model.keras'), monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

print(" Fine-tuning model...")
fine_tune = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=callbacks
)

# 10. Evaluasi
val_loss, val_acc = model.evaluate(val_generator)
print(f" Validation Accuracy: {val_acc:.4f}")

# 11. Prediksi Test Set
test_files = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
test_df = pd.DataFrame({'filename': test_files})
test_df['filepath'] = test_df['filename'].apply(lambda x: os.path.join(test_dir, x))

test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    x_col='filepath',
    y_col=None,
    target_size=(224, 224),
    class_mode=None,
    batch_size=32,
    shuffle=False
)

predictions = model.predict(test_generator)
predicted_classes = np.argmax(predictions, axis=1)

submission = pd.DataFrame({
    'filename': test_df['filename'],
    'label': predicted_classes
})
submission.to_csv(os.path.join(base_dir, 'submission.csv'), index=False)
print("submission.csv berhasil dibuat di folder 'dataset'")


Found 3277 validated image filenames belonging to 15 classes.
Found 820 validated image filenames belonging to 15 classes.
🚀 Training model...
Epoch 1/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 873s 8s/step - accuracy: 0.2127 - loss: 2.4169 - val_accuracy: 0.1841 - val_loss: 4.7634
Epoch 2/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 688s 7s/step - accuracy: 0.2731 - loss: 2.1851 - val_accuracy: 0.1451 - val_loss: 8.9258
Epoch 3/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 685s 7s/step - accuracy: 0.2890 - loss: 2.0983 - val_accuracy: 0.1549 - val_loss: 4.3209
Epoch 4/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 1852s 18s/step - accuracy: 0.3183 - loss: 2.0236 - val_accuracy: 0.2866 - val_loss: 2.4082
Epoch 5/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 848s 8s/step - accuracy: 0.3540 - loss: 1.9387 - val_accuracy: 0.1780 - val_loss: 3.7450
Epoch 6/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 768s 7s/step - accuracy: 0.3741 - loss: 1.8712 - val_accuracy: 0.1805 - val_loss: 5.2245
Epoch 7/25
103/103 ━━━━━━━━━━━━━━━━━━━━ 961s 9s/step - accuracy: 0.3567 - loss: 1